# Study and remove zero-duration training tracks

We inspect `duration_sec` in both original datasets and remove only training rows whose duration is exactly zero. We retain every test track, all other training records, and their original identifiers and relative order.

The cleaned frames remain in memory. We record the findings and removed IDs in `reports/duration_preparation/summary.md`; raw CSV files and existing processed files remain unchanged when this working notebook runs independently.

## 1. Load the original datasets

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = next(
    (directory for directory in (
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd(),
    ) if all((directory / name).is_file() for name in ("train.csv", "test.csv"))),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing train.csv and test.csv.")
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")


## 2. Study duration anomalies

We count zero, missing, and negative durations and inspect the zero-duration training rows. We found two such records, with IDs `244276` and `287253`, and none in the test set.

In [2]:
def summarize_durations(datasets):
    return pd.DataFrame([
        {
            "dataset": name,
            "rows": len(frame),
            "zero-second tracks": int(frame["duration_sec"].eq(0).sum()),
            "% zero-second tracks": round(100 * frame["duration_sec"].eq(0).mean(), 6),
            "missing durations": int(frame["duration_sec"].isna().sum()),
            "negative durations": int(frame["duration_sec"].lt(0).sum()),
        }
        for name, frame in datasets.items()
    ])


def remove_zero_duration_training_rows(frame):
    zero_duration = frame["duration_sec"].eq(0).fillna(False)
    return frame.loc[~zero_duration].copy()

duration_summary = summarize_durations({"train": train_raw, "test": test_raw})
display(duration_summary)
zero_duration_training = train_raw.loc[train_raw["duration_sec"].eq(0)]
removed_training_ids = zero_duration_training["id"].tolist()
display(zero_duration_training[["id", "track_name", "artist_name", "duration_sec", "music_hit"]])


,dataset,rows,zero-second tracks,% zero-second tracks,missing durations,negative durations
0,train,350051,2,0.000571,0,0
1,test,87513,0,0.000000,0,0


,id,track_name,artist_name,duration_sec,music_hit
244275,244276,NaN,Crate Diggers,0.0,0
287252,287253,NaN,Manish Tiwari,0.0,0


## 3. Remove only zero-duration training rows

We exclude the two training records because zero-second durations are suspicious and account for approximately 0.0006% of the original training data. We preserve non-zero durations, including very short tracks, and keep every test row. We do not impute duration values or infer a threshold from the target.

In [3]:
train_clean = remove_zero_duration_training_rows(train_raw)
test_clean = test_raw.copy()
expected_train = train_raw.loc[~train_raw["duration_sec"].eq(0).fillna(False)]
pd.testing.assert_frame_equal(train_clean, expected_train)
pd.testing.assert_frame_equal(test_clean, test_raw)
assert not train_clean["duration_sec"].eq(0).any()
assert len(train_raw) - len(train_clean) == len(removed_training_ids)
assert train_clean["id"].tolist() == expected_train["id"].tolist()
display(summarize_durations({"train_clean": train_clean, "test_clean": test_clean}))


,dataset,rows,zero-second tracks,% zero-second tracks,missing durations,negative durations
0,train_clean,350049,0,0.0,0,0
1,test_clean,87513,0,0.0,0,0


## 4. Record our decision

In [4]:
report_dir = DATA_DIR.parent.parent / "reports/duration_preparation"
report_dir.mkdir(parents=True, exist_ok=True)
lines = [
    "# Zero-duration training tracks", "",
    "We examined zero, missing, and negative durations in the original training and test data.", "",
    "| Dataset | Rows | Zero-second tracks | Zero-second tracks (%) | Missing durations | Negative durations |",
    "| --- | ---: | ---: | ---: | ---: | ---: |",
]
for _, row in duration_summary.iterrows():
    lines.append(
        f"| {row['dataset']} | {row['rows']:,} | {row['zero-second tracks']} | "
        f"{row['% zero-second tracks']:.6f} | {row['missing durations']} | {row['negative durations']} |"
    )
lines += [
    "", "## Decision", "",
    f"We removed {len(removed_training_ids)} training tracks with `duration_sec = 0`: "
    + ", ".join(f"`{value}`" for value in removed_training_ids) + ". "
    "A zero-second duration is suspicious metadata, and these records represent a negligible fraction of the training set. "
    "We retained every other training row and kept the original track identifiers and relative row order. "
    "We did not remove short non-zero tracks or apply a broader duration threshold.", "",
    f"Our prepared training set contains {len(train_clean):,} rows. "
    f"We retained all {len(test_clean):,} test rows because each track requires a prediction. "
    "We included this training-only filter before the other preparation steps in `delivery/music_hit_prediction.ipynb`. "
    "The working notebook keeps its copies in memory; raw CSV files remain unchanged.", "",
]
(report_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_dir / 'summary.md'}")


Report saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/duration_preparation/summary.md
